# Live Arb tick replay — Direct Match

Replays a `live_tick_logs/tsmc_ticks_YYYYMMDD.csv` recording (from the Live
Arb tab's Record button) and, for every distinct timestamp in the log,
re-runs the **exact same** Direct Match algorithm the live app uses —
`logic/live_arb_logic.py::scan()`, the same function
`services/live_arb.py`'s background scan loop calls. We import and call that
function directly rather than re-implementing the matching math, so this is
guaranteed to agree with what the app would have shown at that instant, not
an approximation of it.

**Format:** each snapshot the recorder took is a FULL universe dump —
`services/live_tick_log.py` writes one row per tracked warrant/option for
every timestamp, not just the one code that changed on that tick. So every
`ts` group in the CSV already has everything `scan()` needs; there's no
book cache to reconstruct or carry forward between rows. This notebook is
only compatible with that format — a recording made under the earlier
one-row-per-tick scheme (sparse, only the changed code) will group into
mostly-empty snapshots per timestamp and produce wrong (near-empty) results.
If your file predates this format, re-record and re-run.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root on sys.path so `logic.live_arb_logic` imports exactly as it does
# for app.py/services/live_arb.py — this notebook lives in notebooks/, so the
# repo root is one directory up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from logic import live_arb_logic, iv_engine

print("arb kernel engine:", iv_engine.engine_info())

## Load the tick log

Point `CSV_PATH` at the file you downloaded (or leave it to auto-pick the
newest file under `live_tick_logs/`).

In [ ]:
# Set this explicitly to analyze a specific downloaded file, e.g.:
# CSV_PATH = Path("/path/to/tsmc_ticks_20260903.csv")
CSV_PATH = None

if CSV_PATH is None:
    candidates = sorted((REPO_ROOT / "live_tick_logs").glob("tsmc_ticks_*.csv"))
    if not candidates:
        raise FileNotFoundError("No tsmc_ticks_*.csv found under live_tick_logs/ — record a session first, "
                                 "or set CSV_PATH to a downloaded file.")
    CSV_PATH = candidates[-1]

print("Loading:", CSV_PATH)
raw = pd.read_csv(CSV_PATH, parse_dates=["ts"])
raw["expiry"] = pd.to_datetime(raw["expiry"], errors="coerce").dt.date
raw = raw.sort_values("ts", kind="stable").reset_index(drop=True)

n_snapshots = raw["ts"].nunique()
avg_rows_per_snapshot = len(raw) / n_snapshots if n_snapshots else 0
print(f"{len(raw):,} rows across {n_snapshots:,} distinct snapshot timestamps "
      f"({avg_rows_per_snapshot:.0f} rows/snapshot on average) — {raw['kind'].value_counts().to_dict()}")
if avg_rows_per_snapshot < 5:
    print("WARNING: very few rows per snapshot — this looks like the old sparse (one-row-per-tick) "
          "format, not the full-universe-per-timestamp format this notebook expects. See the note above.")

missing_ratio = raw[(raw["kind"] == "warrant") & raw["exercise_ratio"].isna()]
if len(missing_ratio):
    pct = len(missing_ratio) / (raw["kind"] == "warrant").sum() * 100
    print(f"NOTE: {len(missing_ratio):,} warrant rows ({pct:.0f}%) have no exercise_ratio "
          f"— scan() silently excludes those from matching (a warrant with unresolved terms yet).")

raw.head()

## Replay: one `scan()` call per snapshot timestamp

Each `ts` group already is a full-universe snapshot, so this just reshapes
every group's rows into the dicts `live_arb_logic.scan()`/`build_direct_arrays`
expect (the same shape `services/live_warrant.py::snapshot_for_underlying`/
`services/live_options.py::snapshot_for_underlying` hand to `scan()` in
production) and calls it directly — no book-cache reconstruction needed.

Runtime is dominated by the number of distinct snapshots × one `scan()` call
each (`direct_pairs` itself measures ~8ms for TSMC's full universe per
`logic/live_arb_logic.py`'s docstring). Set `STRIDE > 1` below to only
replay every Nth snapshot for a quicker, coarser look first.

In [ ]:
def _clean(v):
    """NaN/NaT -> None; everything else passed through unchanged. build_direct_arrays
    treats a present-but-falsy value differently from a missing one (e.g. dte<=0 is a
    real drop reason, a missing maturity is a different one), so this must produce a
    real None, never a NaN that happens to be truthy."""
    if v is None or v is pd.NaT:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    return v


def _row_to_warrant(r):
    return {
        "code": r.code,
        "name": _clean(r.name) or r.code,
        "type": _clean(r.type),
        "strike": _clean(r.strike),
        "exercise_ratio": _clean(r.exercise_ratio),
        "maturity": _clean(r.expiry),
        "best": {
            "bid": _clean(r.bid), "ask": _clean(r.ask),
            "bid_size": _clean(r.bid_size), "ask_size": _clean(r.ask_size),
        },
    }


def _row_to_option(r):
    return {
        "code": r.code,
        "name": _clean(r.name) or r.code,
        "type": _clean(r.type),
        "strike": _clean(r.strike),
        "expiry": _clean(r.expiry),
        "best": {
            "bid": _clean(r.bid), "ask": _clean(r.ask),
            "bid_size": _clean(r.bid_size), "ask_size": _clean(r.ask_size),
        },
    }


def replay_direct_match(df, stride=1, progress_every=200):
    groups = list(df.groupby("ts", sort=True))
    if stride > 1:
        groups = groups[::stride]
    n = len(groups)
    out = []
    for i, (ts, group) in enumerate(groups):
        warrant_rows = [_row_to_warrant(r) for r in group[group["kind"] == "warrant"].itertuples(index=False)]
        option_rows = [_row_to_option(r) for r in group[group["kind"] == "option"].itertuples(index=False)]
        hits = live_arb_logic.scan(warrant_rows, option_rows, pd.Timestamp(ts).date())
        for h in hits:
            out.append({"tick_ts": ts, **h})

        if progress_every and (i + 1) % progress_every == 0:
            print(f"{i + 1:,}/{n:,} snapshots replayed, {len(out):,} hit-rows so far")

    return pd.DataFrame(out)


STRIDE = 1  # increase to e.g. 10 for a faster, coarser first pass
hits_df = replay_direct_match(raw, stride=STRIDE)
print(f"\n{len(hits_df):,} (snapshot, pair) hit-rows across {raw['ts'].nunique():,} distinct snapshots")
hits_df.head()

## Quick summary

`hits_df` has one row per (snapshot, matched warrant/option pair) — exactly
`scan()`'s own output fields (`price_diff`, `price_diff_pct`, `riskless`,
strikes, DTEs, ...) plus which snapshot timestamp produced it. A snapshot
with no active pair contributes no rows, so gaps in `tick_ts` are "nothing
was arbable then," not missing data.

In [ ]:
if len(hits_df):
    pair_counts = (hits_df.groupby(["warrant_code", "option_code"])
                   .agg(n_ticks=("tick_ts", "count"),
                        max_price_diff=("price_diff", "max"),
                        max_price_diff_pct=("price_diff_pct", "max"),
                        first_seen=("tick_ts", "min"),
                        last_seen=("tick_ts", "max"))
                   .sort_values("n_ticks", ascending=False))
    display(pair_counts.head(20))

    per_tick_count = hits_df.groupby("tick_ts").size()
    print(f"\nPairs active on {len(per_tick_count):,} distinct ticks "
          f"(max {per_tick_count.max()} simultaneous pairs, "
          f"riskless on {hits_df['riskless'].sum():,}/{len(hits_df):,} hit-rows)")
else:
    print("No active pairs found across the whole replay — check the exercise_ratio "
          "warning above if this file predates that CSV column.")

## Save

Writes the full per-tick hit table next to the source CSV for further
analysis outside this notebook.

In [ ]:
out_path = CSV_PATH.with_name(CSV_PATH.stem + "_direct_match_hits.csv")
hits_df.to_csv(out_path, index=False)
print("Saved:", out_path)